<a href="https://colab.research.google.com/github/apexprit/deception-analysis/blob/main/colab_deception_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🕵️‍♂️ Explainable Multimodal Deception Analysis - Google Colab

**Run this entire deception detection system in the cloud - no local installation required!**

This Colab notebook provides a complete cloud-based deployment of the Explainable Multimodal Deception Analysis System. It detects deception from video interviews using:
- **Facial micro-expressions** (eye movements, mouth dynamics, brow tension)
- **Audio speech patterns** (pitch variation, pauses, speech energy)
- **Temporal dynamics** (probability trajectories, suspicious intervals)
- **SHAP-based explainability** (understand why predictions were made)
- **Subject-adaptive calibration** (personalized baselines)

## 🚀 Quick Start
1. **Runtime → Run all** (Ctrl+F9) to execute everything automatically
2. **Or run cells individually** by clicking the play button ▶️
3. **Upload a video** in the "Analyze Your Own Video" section
4. **Get deception analysis** with visualizations and explanations

## 📊 What You Get
- ✅ **Deception probability** (0-1 scale)
- ✅ **Truthful/Deceptive classification**
- ✅ **Key behavioral indicators**
- ✅ **Temporal analysis plots**
- ✅ **Feature importance explanations**
- ✅ **Publication-quality visualizations**

*No real data needed - synthetic data generation included!*

## 1. Installation & Setup

In [ ]:
#@title Install Dependencies
!pip install -q numpy pandas scikit-learn matplotlib seaborn tqdm
!pip install -q opencv-python mediapipe librosa gradio
!pip install -q shap scipy joblib

print("✅ All dependencies installed successfully!")

In [ ]:
#@title Clone Repository & Setup
import os
import sys
import shutil

# Remove existing directory if present
if os.path.exists('/content/deception-analysis'):
    shutil.rmtree('/content/deception-analysis')

# Create project structure
!git clone -q https://github.com/apexprit/deception-analysis.git /content/deception-analysis

# Add to Python path
sys.path.insert(0, '/content/deception-analysis')

print("✅ Project cloned and set up successfully!")
print(f"Project directory: /content/deception-analysis")

## 2. Generate Synthetic Data (No Real Data Needed)

In [ ]:
#@title Create Synthetic Deception Dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import our synthetic data generator
sys.path.insert(0, '/content/deception-analysis')
from src.utils.synthetic_data import SyntheticDataGenerator
from src.utils.config import get_default_config

# Initialize generator
config = get_default_config()
generator = SyntheticDataGenerator(config, seed=42)

# Generate dataset
print("🔧 Generating synthetic deception dataset...")
features, labels, subject_ids = generator.generate_dataset(
    n_truthful=500,
    n_deceptive=500
)

print(f"✅ Generated {len(features)} samples")
print(f"   Truthful: {sum(labels == 0)} samples")
print(f"   Deceptive: {sum(labels == 1)} samples")
print(f"   Features: {len(features.columns)} (facial, audio, cross-modal)")
print(f"   Unique subjects: {len(set(subject_ids))}")

# Save for later use
os.makedirs('/content/deception-analysis/data/synthetic', exist_ok=True)
features.to_csv('/content/deception-analysis/data/synthetic/features.csv', index=False)
pd.DataFrame({'label': labels, 'subject_id': subject_ids}).to_csv('/content/deception-analysis/data/synthetic/labels.csv', index=False)

print("\n📁 Dataset saved to: /content/deception-analysis/data/synthetic/")

In [ ]:
#@title Visualize Synthetic Data
# Compare key deception indicators
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Select key features
key_features = [
    'eye_left_openness',  # Less eye openness in deception
    'pitch_std',          # Higher pitch variation in deception
    'pause_count',        # More pauses in deception
    'brow_tension',       # Higher brow tension in deception
    'mouth_asymmetry',    # More mouth asymmetry in deception
    'audio_visual_sync'   # Less audio-visual sync in deception
]

# Separate truthful and deceptive samples
truthful_mask = labels == 0
deceptive_mask = labels == 1

for idx, feature in enumerate(key_features):
    ax = axes[idx // 3, idx % 3]
    
    truthful_vals = features.loc[truthful_mask, feature].values
    deceptive_vals = features.loc[deceptive_mask, feature].values
    
    # Plot distributions
    ax.hist(truthful_vals, alpha=0.6, bins=20, label='Truthful', color='green', density=True)
    ax.hist(deceptive_vals, alpha=0.6, bins=20, label='Deceptive', color='red', density=True)
    
    ax.set_title(feature.replace('_', ' ').title())
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    ax.legend()
    
    # Add mean lines
    ax.axvline(np.mean(truthful_vals), color='green', linestyle='--', alpha=0.8)
    ax.axvline(np.mean(deceptive_vals), color='red', linestyle='--', alpha=0.8)

plt.suptitle('Deception Indicators: Truthful (Green) vs. Deceptive (Red) Distributions', 
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("📊 Research-based deception patterns visible in synthetic data:")
print("   • Deceptive: Less eye openness, more pauses, higher pitch variation")
print("   • Deceptive: More brow tension, facial asymmetry, less audio-visual sync")

## 3. Train Deception Detection Model

In [ ]:
#@title Train Model on Synthetic Data
from sklearn.model_selection import train_test_split
from src.model.classifier import DeceptionClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"📈 Training dataset: {len(X_train)} samples")
print(f"📊 Test dataset: {len(X_test)} samples")

# Initialize and train classifier
classifier = DeceptionClassifier(config.model)
print("\n🔧 Training deception classifier...")
train_result = classifier.train(X_train, y_train)

# Evaluate
y_pred = classifier.predict(X_test)
y_prob = classifier.predict_proba(X_test)[:, 1]

test_accuracy = accuracy_score(y_test, y_pred)
test_auc = roc_auc_score(y_test, y_prob)

print("✅ Training complete!")
print(f"\n📊 Performance Metrics:")
print(f"   Training Accuracy: {train_result['accuracy']:.3f}")
print(f"   Training F1-Score: {train_result['f1']:.3f}")
print(f"   Test Accuracy: {test_accuracy:.3f}")
print(f"   Test AUC-ROC: {test_auc:.3f}")

print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Truthful', 'Deceptive']))

In [ ]:
#@title Feature Importance Analysis
# Get feature importance
importance = classifier.get_feature_importance()
importance_df = pd.DataFrame({
    'feature': list(importance.keys()),
    'importance': list(importance.values())
}).sort_values('importance', ascending=False)

# Plot top 15 features
plt.figure(figsize=(12, 6))
top_features = importance_df.head(15)
bars = plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Feature Importance')
plt.title('Top 15 Most Important Features for Deception Detection')

# Color by feature type
for i, (idx, row) in enumerate(top_features.iterrows()):
    feature = row['feature']
    if any(f in feature for f in ['eye', 'mouth', 'brow', 'nose', 'head']):
        bars[i].set_color('skyblue')  # Facial
    elif any(f in feature for f in ['pitch', 'pause', 'energy', 'mfcc', 'spectral']):
        bars[i].set_color('lightcoral')  # Audio
    else:
        bars[i].set_color('lightgreen')  # Cross-modal

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='skyblue', label='Facial Features'),
    Patch(facecolor='lightcoral', label='Audio Features'),
    Patch(facecolor='lightgreen', label='Cross-modal Features')
]
plt.legend(handles=legend_elements, loc='lower right')

plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("🔍 Top 5 deception indicators:")
for i, row in importance_df.head(5).iterrows():
    print(f"   {row['feature']}: {row['importance']:.4f}")

## 4. Launch Gradio Web Interface

In [ ]:
#@title Start Gradio Web App
import sys
sys.path.insert(0, '/content/deception-analysis')

# First, save the trained model
os.makedirs('/content/deception-analysis/models', exist_ok=True)
model_path = '/content/deception-analysis/models/deception_model.pkl'
classifier.save(model_path)
print(f"✅ Model saved to: {model_path}")

# Import and launch Gradio app
print("\n🚀 Launching Gradio web interface...")
print("   This may take a moment to start...")

import gradio as gr
from src.pipeline import DeceptionPipeline

# Initialize pipeline
pipeline = DeceptionPipeline(get_default_config())
pipeline.load_model(model_path)

def analyze_video_proxy(video_path, subject_id):
    if not video_path:
        return "Please upload a video file.", None, None
    
    # Run analysis
    result = pipeline.analyze_video(
        video_path=video_path,
        subject_id=subject_id,
        generate_visualizations=True,
        output_dir='/content/results'
    )
    
    summary = f"### Analysis Results\n"
    summary += f"- **Prediction**: {result['prediction'].upper()}\n"
    summary += f"- **Deception Probability**: {result['deception_probability']:.3f}\n"
    summary += f"- **Confidence**: {result['confidence']:.3f}\n"
    
    viz_paths = result.get('visualizations', {})
    return summary, viz_paths.get('temporal_trajectory'), viz_paths.get('shap_summary')

iface = gr.Interface(
    fn=analyze_video_proxy,
    inputs=[
        gr.Video(label="Upload Interview Video"),
        gr.Textbox(label="Subject ID (Optional)")
    ],
    outputs=[
        gr.Markdown(label="Summary"),
        gr.Image(label="Temporal Trajectory"),
        gr.Image(label="SHAP Explanation")
    ],
    title="🕵️‍♂️ Multimodal Deception Analysis",
    description="Analyze video interviews for deception using facial micro-expressions and audio patterns."
)

iface.launch(share=True, debug=True)